In [1]:
import numpy as np
import pandas as pd
np.random.seed(42)
X=np.random.uniform(-2,2, (400,3))
y= (
np.sin(X[:,0])+
0.5* (X[:,1]**2)+
0.8*X[:,2]
)
y=y.reshape(-1,1)

In [2]:
# ReLU
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)


# sigmoid
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)


# Tanh
def tanh(z):
    return np.tanh(z)

def tanh_derivative(z):
    return 1 - np.tanh(z)**2


# Leaky ReLU
def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha*z)

def leaky_relu_derivative(z, alpha=0.01):
    return np.where(z > 0, 1, alpha)


# Softplus
def softplus(z):
    return np.log(1 + np.exp(z))

def softplus_derivative(z):
    return sigmoid(z)

In [3]:
def count_param(layers):
    total = 0
    for i in range(len(layers)-1):
        n_in = layers[i]
        n_out = layers[i+1]

        weights = n_in * n_out
        biases = n_out
        layer_total = weights + biases

        print(f"Layer {i+1}: ({n_in} → {n_out})")
        print(f"  Weights = {n_in} × {n_out} = {weights}")
        print(f"  Biases  = {biases}")
        print(f"  Total   = {layer_total}\n")

        total += layer_total

    print("Total Parameters =", total)
    return total


In [4]:
# model A
layers_A = [3,4,1]
count_param(layers_A)

# model B
layers_B = [3,6,6,4,1]
count_param(layers_B)

# model C
layers_C = [3,8,8,8,8,1]
count_param(layers_C)

# model D
layers_D = [3,8,8,8,8,8,8,8,8,1]
count_param(layers_D)

Layer 1: (3 → 4)
  Weights = 3 × 4 = 12
  Biases  = 4
  Total   = 16

Layer 2: (4 → 1)
  Weights = 4 × 1 = 4
  Biases  = 1
  Total   = 5

Total Parameters = 21
Layer 1: (3 → 6)
  Weights = 3 × 6 = 18
  Biases  = 6
  Total   = 24

Layer 2: (6 → 6)
  Weights = 6 × 6 = 36
  Biases  = 6
  Total   = 42

Layer 3: (6 → 4)
  Weights = 6 × 4 = 24
  Biases  = 4
  Total   = 28

Layer 4: (4 → 1)
  Weights = 4 × 1 = 4
  Biases  = 1
  Total   = 5

Total Parameters = 99
Layer 1: (3 → 8)
  Weights = 3 × 8 = 24
  Biases  = 8
  Total   = 32

Layer 2: (8 → 8)
  Weights = 8 × 8 = 64
  Biases  = 8
  Total   = 72

Layer 3: (8 → 8)
  Weights = 8 × 8 = 64
  Biases  = 8
  Total   = 72

Layer 4: (8 → 8)
  Weights = 8 × 8 = 64
  Biases  = 8
  Total   = 72

Layer 5: (8 → 1)
  Weights = 8 × 1 = 8
  Biases  = 1
  Total   = 9

Total Parameters = 257
Layer 1: (3 → 8)
  Weights = 3 × 8 = 24
  Biases  = 8
  Total   = 32

Layer 2: (8 → 8)
  Weights = 8 × 8 = 64
  Biases  = 8
  Total   = 72

Layer 3: (8 → 8)
  Weights = 

545

In [5]:
def initialize(layers):
    params = {}
    for l in range(1, len(layers)): # l goes from 1 to L
        params['W'+str(l)] = np.random.randn(layers[l-1], layers[l]) * 0.1
        params['b'+str(l)] = np.zeros((1, layers[l]))
    return params


def forward(X, params, activation):

    cache = {}
    cache['A0'] = X
    L = len(params) // 2

    for l in range(1, L+1):

        W = params['W' + str(l)]
        b = params['b' + str(l)]

        Z = np.dot(cache['A' + str(l-1)], W) + b
        cache['Z' + str(l)] = Z

        if l == L:     # Output layer (linear)
            A = Z
        else:
            A = activation(Z)

        cache['A' + str(l)] = A

    return A, cache
## loss fun
def compute_loss(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

In [10]:
def backward(params, cache, y, activation_derivative, lr=0.01):

    grads = {}
    L = len(params)//2
    m = y.shape[0]

    dA_prev_layer = (cache['A'+str(L)] - y) * 2/m

    for l in reversed(range(1, L+1)):
        Z_l = cache['Z'+str(l)]
        A_l_minus_1 = cache['A'+str(l-1)]
        W_l = params['W'+str(l)]

        if l == L: # Output layer
            dZ = dA_prev_layer
        else:
            dZ = dA_prev_layer * activation_derivative(Z_l)


        grads['dW'+str(l)] = np.dot(A_l_minus_1.T, dZ)
        grads['db'+str(l)] = np.sum(dZ, axis=0, keepdims=True)


        dA_prev_layer = np.dot(dZ, W_l.T)


        params['W'+str(l)] -= lr * grads['dW'+str(l)]
        params['b'+str(l)] -= lr * grads['db'+str(l)]

    return params, grads

def train(X, y, layer_dims, activation, activation_derivative,
          learning_rate=0.01, epochs=1000):

    params = initialize(layer_dims)

    for epoch in range(epochs):
        A, cache = forward(X, params, activation)
        m = y.shape[0]
        loss = (1/m) * np.sum((y - A)**2)

        params, grads = backward(params, cache, y, activation_derivative, learning_rate)

        L_num_layers = len(layer_dims) - 1

        if epoch == epochs - 1:
            print("final loss:", loss)
            grad_norm_L1 = np.sqrt(np.sum(grads["dW1"]**2))
            grad_norm_last = np.sqrt(np.sum(grads["dW" + str(L_num_layers)]**2))
            print("grad norm (L1):", grad_norm_L1)
            print("grad norm (Last hidden layer):", grad_norm_last)

    return loss, grad_norm_L1, grad_norm_last

In [7]:
# model A
model_A = [3, 4, 1]

# model B
model_B =[3,6,6,4,1]

# model C
model_C =[3,8,8,8,8,1]

# model D
model_D = [3,8,8,8,8,8,8,8,8,1]

In [11]:
results = []

print("model A ReLU")
loss, g1, glast = train(X, y, model_A, relu, relu_derivative)
results.append(["A", str(model_A), "ReLU", count_param(model_A), loss, g1, glast])


print("model B ReLU")
loss, g1, glast = train(X, y, model_B, relu, relu_derivative)
results.append(["B", str(model_B), "ReLU", count_param(model_B), loss, g1, glast])


print("model C ReLU")
loss, g1, glast = train(X, y, model_C, relu, relu_derivative)
results.append(["C", str(model_C), "ReLU", count_param(model_C), loss, g1, glast])


print("model D ReLU")
loss, g1, glast = train(X, y, model_D, relu, relu_derivative)
results.append(["D", str(model_D), "ReLU", count_param(model_D), loss, g1, glast])


print("model D Sigmoid")
loss, g1, glast = train(X, y, model_D, sigmoid, sigmoid_derivative)
results.append(["D", str(model_D), "Sigmoid", count_param(model_D), loss, g1, glast])

model A ReLU
final loss: 0.39340765116410703
grad norm (L1): 0.017252156059058842
grad norm (Last hidden layer): 0.00872307621077669
Layer 1: (3 → 4)
  Weights = 3 × 4 = 12
  Biases  = 4
  Total   = 16

Layer 2: (4 → 1)
  Weights = 4 × 1 = 4
  Biases  = 1
  Total   = 5

Total Parameters = 21
model B ReLU
final loss: 1.5006655356472518
grad norm (L1): 0.33594558202471836
grad norm (Last hidden layer): 0.512097700696422
Layer 1: (3 → 6)
  Weights = 3 × 6 = 18
  Biases  = 6
  Total   = 24

Layer 2: (6 → 6)
  Weights = 6 × 6 = 36
  Biases  = 6
  Total   = 42

Layer 3: (6 → 4)
  Weights = 6 × 4 = 24
  Biases  = 4
  Total   = 28

Layer 4: (4 → 1)
  Weights = 4 × 1 = 4
  Biases  = 1
  Total   = 5

Total Parameters = 99
model C ReLU
final loss: 1.8870289864363552
grad norm (L1): 0.08457174506114457
grad norm (Last hidden layer): 0.11071963440806395
Layer 1: (3 → 8)
  Weights = 3 × 8 = 24
  Biases  = 8
  Total   = 32

Layer 2: (8 → 8)
  Weights = 8 × 8 = 64
  Biases  = 8
  Total   = 72

Layer 3

In [15]:
import pandas as pd

columns = [
    "Model",
    "Layers",
    "Activation",
    "Total Parameters",
    "Final Loss",
    "Grad Norm (Layer 1)",
    "Grad Norm (Last Hidden)"
]

observation_table = pd.DataFrame(results, columns=columns)

print(observation_table.round(6))
observation_table.to_csv("observation_table.csv", index=False)

  Model                          Layers Activation  Total Parameters  \
0     A                       [3, 4, 1]       ReLU                21   
1     B                 [3, 6, 6, 4, 1]       ReLU                99   
2     C              [3, 8, 8, 8, 8, 1]       ReLU               257   
3     D  [3, 8, 8, 8, 8, 8, 8, 8, 8, 1]       ReLU               545   
4     D  [3, 8, 8, 8, 8, 8, 8, 8, 8, 1]    Sigmoid               545   

   Final Loss  Grad Norm (Layer 1)  Grad Norm (Last Hidden)  
0    0.393408             0.017252                 0.008723  
1    1.500666             0.335946                 0.512098  
2    1.887029             0.084572                 0.110720  
3    1.946979             0.000001                 0.000001  
4    1.946979             0.000000                 0.000000  
